# Exploratory GO enrichment of consensus gene panels

**Exploratory, hypothesis-generating only.** Panels are the genes selected in at least `MIN_FREQ` of
the outer **training** folds (never genes chosen with test data). Affymetrix probes are mapped to
gene symbols with the platform annotation; the background is every gene measured on the array.
Test: one-sided hypergeometric per GO set, Benjamini-Hochberg FDR.

Inputs (not in git): `data/<dataset>.csv`, a platform annotation such as `data/ref/GPL96.annot.gz`
(GEO: `ftp.ncbi.nlm.nih.gov/geo/platforms/GPLnnn/GPL96/annot/`), and a GMT library such as
`data/ref/GO_Biological_Process_2023.gmt` (Enrichr library download).

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
%matplotlib inline

In [ ]:
DATASET = os.environ.get("GWO_DATASET", "Leukemia_GSE9476")
RESULTS_DIR = ROOT / os.environ.get("GWO_RESULTS_DIR", "results")
DATA_PATH = ROOT / "data" / f"{DATASET}.csv"
ANNOTATION = ROOT / os.environ.get("GWO_ANNOTATION", "data/ref/GPL96.annot.gz")
GMT = ROOT / os.environ.get("GWO_GMT", "data/ref/GO_Biological_Process_2023.gmt")
K = int(os.environ.get("GWO_K", 30))
MIN_FREQ = 0.6

In [ ]:
import pandas as pd
from src import enrichment, tables
from src.data import load_cumida

df = tables.read_results(tables.results_path(DATASET, RESULTS_DIR))
_, _, genes = load_cumida(DATA_PATH)
mapping = enrichment.load_probe_annotation(ANNOTATION) if enrichment.looks_like_probe_ids(genes) else None
background, n_unmapped = enrichment.map_to_symbols(genes, mapping)
gene_sets = enrichment.load_gmt(GMT)
print(f"background: {len(background)} symbols ({n_unmapped} unmapped probes); {len(gene_sets)} gene sets")

In [ ]:
out = RESULTS_DIR / "tables"
out.mkdir(parents=True, exist_ok=True)
rows, results = [], {}
for sel in sorted(df.loc[df["n_genes"] == df["k"], "selector"].unique()):
    cons = enrichment.consensus_genes(df, sel, genes, k=K, dataset=DATASET, min_freq=MIN_FREQ)
    symbols, unmapped = enrichment.map_to_symbols(cons.index, mapping)
    res = enrichment.ora(symbols, background, gene_sets)
    res.to_csv(out / f"enrichment_{DATASET}_{sel}.csv", index=False)
    results[sel] = res
    rows.append({"selector": sel, "consensus_probes": len(cons), "symbols": len(symbols),
                 "unmapped": unmapped, "sets_tested": len(res),
                 "sets_q<0.05": int((res["q_value"] < 0.05).sum()),
                 "top_set": res["gene_set"].iloc[0] if len(res) else "", "top_q": res["q_value"].min()})
summary = pd.DataFrame(rows)
(out / f"enrichment_summary_{DATASET}.md").write_text(tables.to_markdown(summary) + "\n", encoding="utf-8")
summary

In [ ]:
pd.concat({s: r.head(5)[["gene_set", "overlap", "set_size", "fold_enrichment", "p_value", "q_value", "genes"]]
           for s, r in results.items()}, names=["selector", "rank"])

Optional: Enrichr via gseapy (needs `pip install gseapy` and network access).

In [ ]:
try:
    sel = "ANOVA"
    cons = enrichment.consensus_genes(df, sel, genes, k=K, dataset=DATASET, min_freq=MIN_FREQ)
    symbols, _ = enrichment.map_to_symbols(cons.index, mapping)
    enrichment.run_enrichr(symbols, background=sorted(background)).head()
except ImportError as exc:
    print(exc)

## Caveats (carry into the report)

- Interpretation is **exploratory**; with panels of ~10-30 genes few sets survive FDR.
- Leukemia vs. normal blood/marrow contrasts are confounded by **cell-type composition / immune
  infiltration**: enriched immune terms may reflect which cells are in the sample, not tumour biology.
- **Batch effects** (scan date, lab, processing) can drive selection if correlated with class.
- **Probe-level artifacts**: cross-hybridising `_x_at`/`_s_at` probes, multi-gene probes (`///`) and
  unmapped probes distort the gene-level picture.